In [1]:
def show_isosurface(sim, iso=None, title="3D Plasma Isosurface"):
    """
    Improved isosurface renderer that auto-selects a visible iso value.
    If iso is None or produces no surface, it automatically searches for one.
    """

    P = sim.P.copy()
    P = (P - P.min()) / (P.max() - P.min() + 1e-12)  # Normalize to [0,1]

    # Auto-select iso if None or if iso is too high (empty surface risk)
    if iso is None:
        iso = np.percentile(P, 85)     # auto choose top 15%
    if iso >= 0.95:
        iso = 0.85                     # push lower for visibility

    # Check whether surface exists at chosen iso
    mask = P >= iso
    if np.sum(mask) < 50:  # too few points to render
        # fallback search: try decreasing levels until visible
        for level in [0.70, 0.60, 0.50, 0.40, 0.30, 0.20, 0.10]:
            if np.sum(P >= level) > 200:
                print(f"[Auto-adjust] Iso {iso:.2f} too high -> using {level:.2f}")
                iso = level
                break
        else:
            print("⚠️ Warning: No isosurface found for any threshold — field too flat.")
            return

    print(f"Rendering isosurface at iso={iso:.2f} ...")

    if HAS_PLOTLY:
        fig = go.Figure(data=go.Isosurface(
            x=sim.X.flatten(), y=sim.Y.flatten(), z=sim.Z.flatten(),
            value=P.flatten(),
            isomin=iso, isomax=iso,
            surface_count=1,
            caps=dict(x_show=False, y_show=False, z_show=False),
            colorscale="Turbo"
        ))
        fig.update_layout(
            title=f"{title} (iso={iso:.2f})",
            scene=dict(xaxis_title="x", yaxis_title="y", zaxis_title="z", aspectmode="data"),
            width=900, height=700
        )
        fig.show()

    else:
        # Fallback: scatter
        mask = P >= iso
        x, y, z = sim.X[mask], sim.Y[mask], sim.Z[mask]
        if x.size > 40000:
            idx = np.random.choice(x.size, 40000, replace=False)
            x, y, z = x.flatten()[idx], y.flatten()[idx], z.flatten()[idx]

        fig = plt.figure(figsize=(8,7))
        ax = fig.add_subplot(111, projection='3d')
        ax.scatter(x, y, z, s=1, alpha=0.35)
        ax.set_title(f"{title} (iso={iso:.2f})")
        ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")
        ax.view_init(22, 35)
        plt.show()